# hoodini CLI Workflow Demo

This notebook demonstrates the complete **hoodini** pipeline for analyzing genomic neighborhoods (gene contexts) across bacterial/archaeal genomes.

## What does hoodini do?

**hoodini** helps you:
1. **Find** homologs of your query protein(s) across NCBI genomes
2. **Extract** genomic neighborhoods around those homologs
3. **Annotate** genes with functional predictions (domains, defense systems, CRISPR, etc.)
4. **Compare** neighborhoods using protein clustering and sequence alignments
5. **Visualize** results in an interactive HTML viewer

## Pipeline Overview

The workflow follows these stages:
- **Input & Search**: Parse query sequences, search NCBI for homologs
- **Neighborhood Extraction**: Download genomes and extract gene contexts
- **Clustering**: Group similar proteins into families
- **Phylogeny/Taxonomy**: Build evolutionary tree of genomes
- **Optional Links**: Compute protein-protein and/or nucleotide-nucleotide similarities
- **Extra Annotations**: Add domains, CRISPR arrays, ncRNAs, defense systems, etc.
- **Export**: Generate interactive visualizations and data files

Each cell below corresponds to one stage of the pipeline.

## 1. Import Required Libraries
Import all necessary libraries and modules used in cli.py, including any custom modules.

In [1]:
# Patch asyncio for Jupyter compatibility
from pathlib import Path

import nest_asyncio
import polars as pl

nest_asyncio.apply()

## 3. Configure Pipeline Parameters

Set all parameters for the pipeline run. **Edit these values** to customize your analysis.

---

### 📝 **Input Formats** (`input_path` or `inputsheet`)

hoodini accepts **3 types of input**:

#### **Option 1: List of IDs** (most common)

A text file with one ID per line. Supported ID types:

| ID Type | Examples | Description |
|---------|----------|-------------|
| **NCBI Protein** | `WP_000001234.1`, `NP_123456.1`, `YP_789012.1` | RefSeq protein accessions |
| **NCBI Nucleotide** | `NC_000913.3`, `NZ_CP012345.1`, `LR123456.1` | Chromosome/contig accessions |
| **UniProt** | `P12345`, `Q9UHC1`, `A0A0A0MRZ7` | UniProt accessions (auto-converted to NCBI) |
| **Nucleotide with coordinates** | `NC_000913.3:1000-2000` | Specific genomic region |

```text
# Example: cas9.txt
WP_010922251.1
WP_002989955.1
NP_472073.1
```

#### **Option 2: Single Query ID**

If `input_path` is a **single ID** (not a file), hoodini will:
1. Fetch the sequence from NCBI/UniProt
2. Run remote BLAST to find homologs
3. Use all hits as input for neighborhood extraction

```python
input_path = "WP_010922251.1"  # Single ID triggers remote BLAST
```

#### **Option 3: Input Sheet** (advanced, skip homolog search)

A TSV file with pre-defined genomic locations. Use when you already know where your genes are.

**Required columns**: `protein_id`, `nucleotide_id`, `gff_path`, `fna_path`, `faa_path`

```tsv
protein_id	nucleotide_id	gff_path	fna_path	faa_path
WP_000001234.1	NC_000913.3	/path/to/genome.gff	/path/to/genome.fna	/path/to/proteins.faa
```

Use: `inputsheet="my_locations.tsv"` instead of `input_path`

---

### ⚙️ **Other Parameters**

**Output**:
- `output_dir`: Where to save all results
- `assembly_folder`: Local genome folder (optional; if None, downloads from NCBI)

**Tool Toggles** (True/False):
- `ncrna`: Detect non-coding RNAs (tRNAs, rRNAs, riboswitches)
- `cctyper`: Detect CRISPR-Cas systems
- `genomad`: Detect mobile genetic elements (plasmids, phages)
- `domains`: List of domain DBs (e.g., `["pfam", "tigrfam"]`); empty list `[]` skips
- `blast`: Path to custom BLAST database FASTA; `None` skips

**Similarity Networks**:
- `nt_links`: Compute nucleotide alignments between neighborhoods
- `prot_links`: Compute protein-protein similarity links

**Performance**:
- `num_threads`: CPU cores to use
- `max_concurrent_downloads`: Parallel genome downloads

**Window Parameters** (neighborhood size):
- `mod`: `"win_nts"` (nucleotide window) or `"win_genes"` (gene count)
- `wn`: Window size (20000 nt or 10 genes typical)
- `sorfs`: Include small ORFs (<50 aa)
- `minwin`: Minimum neighborhood size filter
- `minwin_type`: `"both"`, `"upstream"`, or `"downstream"`

**Phylogeny**:
- `tree_mode`: `"taxonomy"`, `"aai"` (protein identity), or `"ani"` (nucleotide identity)
- `aai_mode` / `ani_mode`: `"fast"` or `"slow"`
- `nj_algorithm`: `"nj"` (neighbor-joining) or `"upgma"`

**Remote Search** (for single query or BLAST):
- `remote_evalue`: E-value cutoff (default: 1e-5)
- `remote_max_targets`: Max homologs to retrieve
- `cand_mode`: `"all"` or `"representative"` (RefSeq only)

In [2]:
# Manual parameters (edit here); no defaults loaded
input_path = "cas9.txt"
output_dir = "notebook-output"
assembly_folder = None

# Tool toggles
ncrna = True
cctyper = True
genomad = True
domains = ["pfam"]  # list of domain DBs to run; [] skips
blast = None  # path to BLAST database FASTA; set to None to skip

# Link calculations
nt_links = False
prot_links = False

# Performance
num_threads = 8
max_concurrent_downloads = 8

# Window parameters
mod = "win_nts"
wn = 20000
sorfs = False
minwin = None
minwin_type = "both"

# Tree / clustering options
tree_mode = "taxonomy"
tree_file = None
aai_mode = "fast"
ani_mode = "fast"
nt_aln_mode = "fast"
aai_subset_mode = None
nj_algorithm = "nj"
clust_method = "diamond_deepclust"

# Remote search tuning
remote_evalue = 1e-5
remote_max_targets = 100
cand_mode = "all"


## 4. Execute Core Workflow Logic
Run the main logic of the workflow, step by step, as implemented in cli.py.

## 5. Stage 1: Initialize Inputs

**What it does**: Reads your input and prepares records for the pipeline.

### Behavior depends on input type:

| Input Type | What Happens |
|------------|--------------|
| **List of IDs (file)** | Parses each ID, categorizes type (protein/nucleotide/uniprot), fetches sequences if needed |
| **Single ID (literal)** | Fetches sequence → **runs remote BLAST** → uses all hits as input |
| **Input Sheet (TSV)** | Reads pre-defined locations directly, skips homolog search |

### ⚠️ Single Query Mode (Remote BLAST)

If `input_path` is a **single ID** (not a file path), this step will:
1. Fetch the query sequence from NCBI
2. **Run BLASTp against NCBI's `nr_cluster_seq` database**
3. Filter hits by `remote_evalue` and `remote_max_targets`
4. Return all hits as the input for subsequent stages

**⏱️ Note**: Remote BLAST can take **several minutes** depending on NCBI server load and the number of hits. For large-scale analyses, consider pre-computing homologs and using a file input instead.

### ID Auto-Detection:

hoodini automatically detects ID type based on format:
- `WP_`, `NP_`, `YP_`, `XP_`, `ZP_` → NCBI protein
- `NC_`, `NZ_`, `NM_`, `AC_`, etc. → NCBI nucleotide
- `P12345`, `Q9UHC1` format → UniProt (converted to NCBI via mapping)

**Output**: `records` DataFrame with columns:
- `og_index`: Original query index
- `protein_id` / `nucleotide_id`: Parsed IDs
- `input_type`: `"protein"` or `"nucleotide"`
- `assembly_id`, `taxid`: Populated in later stages

In [3]:
from hoodini.pipeline.initialize import initialize_inputs

# Initialize the records DataFrame
records = initialize_inputs(
    input_path=input_path,
    output=output_dir,
    remote_evalue=remote_evalue,
    remote_max_targets=remote_max_targets,
    force=True,
    inputsheet=None,
)


[12:33:18] 📁 Assembly DB found: /home/pentamorfico/software/hoodini/src/hoodini/data/assembly_summary.parquet 
(updated: 2026-01-09)

[12:33:18] ✅ Using existing database (less than 1 month old).

[12:33:18] ✔️  Created folder notebook-output

## 6. Stage 2: Find Homologs in Genomes

**What it does**: Queries NCBI's Identical Protein Groups (IPG) database to find where your query proteins (and their identical copies) exist across all sequenced genomes.

**Process**:
1. Queries IPG database for each protein ID
2. Retrieves all genomic locations where that protein (or identical sequences) are annotated
3. Maps proteins to their genomic assemblies and coordinates
4. Filters by candidate mode (`cand_mode`): `"all"` genomes or `"representative"` RefSeq genomes only

**Important**: This step does **NOT** use BLAST. IPG finds **identical** protein sequences, not similar ones. BLAST is only used in single-query mode (Stage 1) to find homologs before this step.

**Output**:
- `records` DataFrame enriched with assembly IDs, genomic coordinates, and candidate counts

This step determines **which genomes** will be analyzed in the next stage.

In [4]:
from hoodini.pipeline.parse_ipg import run_ipg

records = run_ipg(
    records_df=records,
    cand_mode=cand_mode,
)


[12:33:19] 🔍  Fetching IPG data...

Output()

[12:33:20] Fetched 20 IPG records for 10 proteins.

[12:33:20] 🔍  Fetching nucleotide data...

[12:33:36] ✅  Selecting best IPG records...

## 7. Stage 3: Extract Genomic Neighborhoods

**What it does**: Downloads genome assemblies and extracts gene neighborhoods around query homologs.

**Process**:
1. Downloads GFF3 (gene annotations) and FASTA (sequences) for each assembly
2. Finds query homologs in annotations
3. Extracts surrounding genes based on window parameters (`wn`, `mod`)
4. Parses protein sequences and genomic coordinates

**Outputs**:
- `all_gff`: All genes in GFF format (genomic coordinates + metadata)
- `all_prots`: Protein sequences + annotations (product, coordinates, etc.)
- `all_neigh`: Neighborhood metadata (window boundaries, central gene)
- `valid_uids`: List of successfully processed assemblies

**Window modes**:
- `win_nts=20000`: ±10kb around query
- `win_genes=10`: ±5 genes around query

In [5]:
from hoodini.pipeline.parse_assemblies import run_assembly_parser

result = run_assembly_parser(
    records_df=records,
    output_dir=output_dir,
    assembly_folder=assembly_folder,
    ncrna=ncrna,
    cctyper=cctyper,
    genomad=genomad,
    blast=blast,
    max_concurrent_downloads=max_concurrent_downloads,
    num_threads=num_threads,
    mod=mod,
    wn=wn,
    sorfs=sorfs,
    minwin=minwin,
    minwin_type=minwin_type,
 )

[12:33:37] No cached assemblies found in assembly_folder; downloading all.

[12:33:49] ✔ Downloaded or located assemblies (folder: 
/home/pentamorfico/software/hoodini/example/notebook-output/assembly_folder)

Parsing GBFF:   0%|          | 0/20 [00:00<?, ?it/s]

[12:33:51] Warning: Failed extractions for 8 records: ['9', '5', '2', '11', '13', '15', '18', '19']

[12:33:51] ✔ Extracted neighborhoods and wrote output files.

In [6]:
records = result["records"]
all_gff = result["all_gff"]
all_prots = result["all_prots"]
all_neigh = result["all_neigh"]
valid_uids = result["valid_uids"]

## 8. Stage 4: Cluster Proteins into Families

**What it does**: Groups similar proteins together to simplify visualization.

**Process**:
- Clusters all proteins using sequence similarity (DIAMOND or MMseqs2)
- Assigns each protein a `cluster` ID (family number)

**Why?**: Instead of showing every unique protein, we group similar ones (e.g., "Cas9 family", "DNA helicase family") for cleaner visual comparisons.

**Methods**:
- `diamond_deepclust`: Fast, hierarchical clustering (default)
- `mmseqs_cluster`: Alternative, different sensitivity/speed trade-off

**Output**: `all_prots` updated with `cluster` column.

In [7]:
from hoodini.pipeline.cluster_proteins import cluster_proteins

all_prots = cluster_proteins(
    all_prots,
    output_dir=output_dir,
    clust_method=clust_method,
    sorfs=sorfs,
)


[12:33:52] ✔️    Protein clustering complete

## 9. Stage 5: Build Phylogenetic Tree

**What it does**: Creates a phylogenetic tree showing evolutionary relationships between genomes.

**Tree modes**:
1. **`taxonomy`** (fastest): Uses NCBI taxonomy hierarchy
   - Pro: Instant, biologically meaningful
   - Con: Only reflects known taxonomy, not your specific data

2. **`aai`** (Average Amino acid Identity): Compares all proteins
   - Pro: Data-driven, reflects genome-wide similarity
   - Con: Slower, requires protein alignments

3. **`ani`** (Average Nucleotide Identity): Compares DNA sequences
   - Pro: Strain-level resolution
   - Con: Slowest, only works for closely related genomes

**Outputs**:
- `tree_str`: Newick format tree (standard phylogenetic format)
- `den_data`: Dendrogram metadata (branch lengths, node positions) for visualization

In [8]:
from hoodini.pipeline.taxonomy import parse_taxonomy_and_build_tree

tree_str, den_data = parse_taxonomy_and_build_tree(
    records=records,
    all_gff=all_gff,
    all_neigh=all_neigh,
    all_prots=all_prots,
    output_dir=output_dir,
    tree_mode=tree_mode,
    tree_file=tree_file,
    num_threads=num_threads,
    aai_mode=aai_mode,
    ani_mode=ani_mode,
    aai_subset_mode=aai_subset_mode,
    nj_algorithm=nj_algorithm,
)


[12:33:52] ✔ Tree saved as Newick

## 10. Stage 6: Compute Protein Similarity Links (Optional)

**What it does**: Finds protein-protein similarities **across** neighborhoods.

**When to use**: If you want to highlight shared proteins between different neighborhoods (e.g., "these two neighborhoods share a transposase").

**Process**:
- BLASTs all proteins against each other
- Filters by e-value (default: 1e-5)
- Creates edges (links) between similar proteins

**Output**: `pairwise_aa` DataFrame with protein pairs and similarity scores.

**Skip if**: You only care about within-neighborhood gene order (not cross-neighborhood protein conservation).

In [9]:
from hoodini.pipeline.protein_links import run_protein_links

pairwise_aa = None
if prot_links:
    pairwise_aa = run_protein_links(
        output_dir=output_dir,
        all_prots=all_prots,
        threads=num_threads,
        evalue=1e-5,
    )


## 11. Stage 7: Compute Nucleotide Similarity Links (Optional)

**What it does**: Aligns DNA sequences between neighborhoods to find conserved regions.

**When to use**: If you want to detect conserved intergenic regions, regulatory elements, or assess overall synteny (gene order conservation).

**Process**:
- Aligns full neighborhood DNA sequences (e.g., using minimap2 or BLAST)
- Identifies aligned blocks
- Computes ANI (Average Nucleotide Identity) between neighborhoods

**Outputs**:
- `pairwise_ani`: ANI matrix between all neighborhood pairs
- `nt_links_result`: Alignment blocks for visualization (shows which DNA regions match)

**Note**: Computationally expensive for large datasets. Best for closely related genomes (>70% ANI).

In [10]:
from hoodini.pipeline.pairwise_nt import run_pairwise_nt

pairwise_ani = None
nt_links_result = None
if nt_links:
    pairwise_ani, nt_links_result = run_pairwise_nt(
        all_neigh=all_neigh,
        all_gff=all_gff,
        output_dir=output_dir,
        nt_aln_mode=nt_aln_mode,
        ani_mode=ani_mode,
        nt_links=nt_links,
        threads=num_threads,
    )


## 12. Stage 8: Annotate Protein Domains

**What it does**: Identifies functional domains in proteins (e.g., "DNA-binding domain", "ATPase domain").

**Databases supported**:
- `pfam`: Protein family database (most comprehensive)
- `tigrfam`: Specialized for prokaryotic proteins
- `cog`: Clusters of Orthologous Groups
- `smart`: Simple Modular Architecture Research Tool

**Process**:
- Searches each protein against domain HMM profiles
- Reports domain hits with e-values and positions

**Output**: `domains_data` DataFrame with domain annotations per protein.

**Why useful**: Domains reveal protein function even when full sequence is divergent (e.g., "Has helicase domain → probably unwinds DNA").

In [11]:
from hoodini.extra_tools.domain import run_domain

domains_data = None
if domains:
    domains_data = run_domain(all_prots, output_dir, domains, num_threads)


[12:33:56] 🔍   Annotating domains with HMMER: pfam

[12:33:56] Loading HMMs for pfam (this can take a moment)...

Output()

[12:34:27] ✔️    Domain annotation complete: 335 matches from 1 databases (deduplicated from 652).

## 13. Stage 9: Detect Defense Systems (PADLOC)

**What it does**: Identifies anti-phage defense systems using PADLOC (Prokaryotic Antiviral Defense LOCator).

**Defense systems detected**:
- Restriction-Modification (R-M) systems
- Toxin-Antitoxin (TA) modules
- Abortive infection systems
- Innate immune systems (BREX, DISARM, etc.)
- And many more...

**Process**:
- Searches proteins against PADLOC's HMM profiles
- Identifies multi-component defense systems
- Annotates system type and completeness

**Output**: `all_prots` updated with defense system annotations (columns like `padloc_system`, `padloc_protein`).

**Note**: Especially useful for studying phage-bacteria interactions and immune system evolution.

In [ ]:
from hoodini.extra_tools.padloc import run_padloc

padloc_df = run_padloc(all_gff, all_prots, output_dir, num_threads)
all_prots = pl.DataFrame(all_prots).join(pl.DataFrame(padloc_df), on="id", how="left")


[12:34:31] 🧬   Running PADLOC...

[12:34:31] >> Scanning temp.fasta for defence system proteins


## 14. Stage 10: Detect Defense Systems (DefenseFinder)

**What it does**: Another defense system detector using DefenseFinder database.

**Differences from PADLOC**:
- DefenseFinder: Broader catalog, includes newly discovered systems
- PADLOC: More conservative, well-characterized systems

**Systems detected**:
- CRISPR-associated proteins (beyond Cas proteins)
- Druantia, Hachiman, Lamassu, Zorya, etc.
- Novel defense systems from recent literature

**Process**: Similar to PADLOC but uses different profile database.

**Output**: `all_prots` updated with DefenseFinder annotations.

**Tip**: Running both PADLOC and DefenseFinder gives comprehensive defense system coverage.

In [ ]:
from hoodini.extra_tools.defensefinder import run_defensefinder

deffinder_df = run_defensefinder(all_gff, all_prots, output_dir)
all_prots = pl.DataFrame(all_prots).join(pl.DataFrame(deffinder_df), on="id", how="left")


## 15. Stage 11: Extra Annotations (BLAST, CRISPR, ncRNA, MGEs)

**This stage runs optional extra tools** based on your parameter settings:

### 🔍 **BLAST** (`blast` parameter)
- **What**: Search neighborhoods against custom database
- **When**: You have a known gene/protein and want to find similar sequences
- **Example**: Search for toxin genes, antibiotic resistance markers
- **Output**: `blast_data` with hits → added to GFF as "regions"

### 🧬 **CCTyper** (`cctyper=True`)
- **What**: Detects CRISPR arrays and classifies Cas protein types
- **Finds**: CRISPR arrays (repeats + spacers), Cas subtypes (I-A, II-A, V-A, etc.)
- **Output**: 
  - `cctyper_df`: Cas protein annotations → joined to `all_prots`
  - `crispr_df`: CRISPR array locations → added to GFF

### 🧪 **ncRNA Detection** (`ncrna=True`)
- **What**: Finds non-coding RNAs using Rfam covariance models
- **Detects**: tRNAs, rRNAs, riboswitches, sRNAs, ribozymes
- **Output**: `ncrna_data` with RNA family, position, strand, score → added to GFF and parquet

### 🦠 **geNomad** (`genomad=True`)
- **What**: Identifies mobile genetic elements (MGEs)
- **Detects**: Prophages, plasmids, conjugative transposons, integrative elements
- **Method**: Machine learning classifier trained on MGE features
- **Output**: `genomad_df` with element type and coordinates → added to GFF

**Note**: All these tools' raw outputs are automatically transformed to GFF format by `write_viz_outputs` for unified visualization.

In [ ]:
# Extra tools: BLAST, CCTyper, geNomad, ncRNA
# Run tools and save raw outputs (GFF transformation happens in write_viz_outputs)

blast_data = None
crispr_df = None
genomad_df = None

if blast:
    from hoodini.extra_tools.blast import run_blast
    blast_data = run_blast(all_neigh, output_dir, blast, num_threads, valid_uids)

if cctyper:
    from hoodini.extra_tools.cctyper import run_cctyper
    cctyper_df, crispr_df = run_cctyper(all_gff, all_prots, all_neigh, output_dir, num_threads, valid_uids)
    if cctyper_df.height > 0:
        all_prots = all_prots.join(cctyper_df, on="id", how="left")

if ncrna:
    from hoodini.extra_tools.ncrna import run_ncrna
    ncrna_data = run_ncrna(all_neigh, den_data, output_dir, num_threads, valid_uids)

if genomad:
    from hoodini.extra_tools.genomad import run_genomad
    genomad_df = run_genomad(all_neigh, output_dir, num_threads, valid_uids)


## 16. Stage 12: Export Visualization Files

**What it does**: Packages all analysis results into visualization-ready formats.

**Generated files** in `{output_dir}/hoodini-viz/`:

### 📊 **Parquet files** (for programmatic access):
- `gff.parquet`: All genomic features (genes, ncRNAs, CRISPR arrays, etc.)
- `hoods.parquet`: Neighborhood boundaries and metadata
- `protein_metadata.parquet`: Protein sequences, clusters, domains, defense systems
- `tree_metadata.parquet`: Phylogenetic tree structure and branch data
- `domains.parquet`: Domain annotations (if domains enabled)
- `ncrna_metadata.parquet`: ncRNA details (if ncRNA enabled)
- `nucleotide_links.parquet`: DNA alignment blocks (if nt_links enabled)
- `protein_links.parquet`: Protein similarity edges (if prot_links enabled)

### 📄 **TSV files** (human-readable):
- Same data as parquet but in tab-separated text format
- Useful for quick inspection, Excel, R/Python scripts

### 🌳 **Other outputs**:
- `tree.nwk`: Newick phylogenetic tree (standard format for tree viewers)
- `hoodini-viz.html`: **Standalone interactive HTML viewer**
  - Embeds all data (no external dependencies)
  - Open in any web browser
  - Explore neighborhoods, zoom, compare gene orders
  - Filter by annotations (domains, defense systems, etc.)

**Bonus**: If run in Jupyter, the HTML viewer auto-displays in the notebook output!

In [ ]:
# Write visualization outputs using write_data module
from hoodini.pipeline.write_data import write_viz_outputs

viz_dir = write_viz_outputs(
    output_dir=output_dir,
    all_gff=all_gff,
    all_neigh=all_neigh,
    all_prots=all_prots,
    den_data=den_data,
    tree_str=tree_str,
    records=records,
    nt_links=nt_links_result if 'nt_links_result' in globals() else None,
    pairwise_aa=pairwise_aa if 'pairwise_aa' in globals() else None,
    domains_data=domains_data if 'domains_data' in globals() else None,
    ncrna_data=ncrna_data if 'ncrna_data' in globals() else None,
    write_domains=bool(domains),
    blast_data=blast_data if 'blast_data' in globals() else None,
    crispr_df=crispr_df if 'crispr_df' in globals() else None,
    genomad_df=genomad_df if 'genomad_df' in globals() else None,
)

print(f"Visualization outputs written to: {viz_dir}")

### 🌐 View Interactive Visualization

The cell below displays the visualization directly in the notebook using `IPython.display.HTML`.

**Note**: For very large visualizations (>10MB), you may need to open the HTML file directly in a browser instead.

In [ ]:
# Display visualization directly in notebook
from IPython.display import HTML, display

html_path = Path(viz_dir) / "hoodini-viz.html"
html_content = html_path.read_text(encoding="utf-8")

# Wrap in a container with controlled dimensions
wrapped_html = f"""
<div style="width: 100%; max-width: 100%; height: 800px; overflow: hidden; border: 1px solid #333; border-radius: 4px;">
    <iframe srcdoc="{html_content.replace('"', '&quot;')}" 
            style="width: 100%; height: 100%; border: none;">
    </iframe>
</div>
"""

display(HTML(wrapped_html))

---

## 🎉 Pipeline Complete!

### What You Have Now:

1. ✅ **Genomic neighborhoods** extracted from homolog-containing assemblies
2. ✅ **Protein families** clustered by sequence similarity
3. ✅ **Phylogenetic tree** showing genome relationships
4. ✅ **Functional annotations**: domains, defense systems, CRISPR, ncRNAs, MGEs
5. ✅ **Interactive visualizations** ready to explore

### Next Steps:

**🌐 View Results**:
- Open `{output_dir}/hoodini-viz/hoodini-viz.html` in a browser
- Explore gene synteny (order conservation) across genomes
- Filter/highlight genes by annotation (e.g., show only defense systems)
- Compare neighborhood architectures in phylogenetic context

**📊 Analyze Data**:
```python
import polars as pl

# Load protein metadata
prots = pl.read_parquet(f"{output_dir}/hoodini-viz/parquet/protein_metadata.parquet")

# Example: Find all proteins with "CRISPR" annotations
crispr_prots = prots.filter(pl.col("cctyper_prediction").is_not_null())

# Example: Get domain statistics
domain_counts = prots.group_by("cluster").agg(pl.col("id").count().alias("count"))
```

**🔬 Biological Questions to Ask**:
- Are neighborhoods conserved across phylogeny or mosaic?
- Do certain defense systems co-occur with query genes?
- Are CRISPR spacers targeting mobile elements in the neighborhoods?
- Do ncRNAs show position conservation (potential regulatory role)?

**📖 Documentation**: See `docs/` folder for more details on each pipeline stage and output format.